# ABS Annual Taxation Revenue 5506.0

## Python set-up

In [1]:
# analytic imports
import pandas as pd
import readabs as ra
import mgplot as mg

# pandas display settings
pd.options.display.max_rows = 9999
pd.options.display.max_columns = 999
pd.options.display.max_colwidth = 120

# display charts within this notebook
SHOW = False
RFOOTER = "ABS 5506.0"

# 5506.0 is an Excel-only release (not in the ABS Time Series Directory),
# so we hit the landing page directly.
TAX_URL = (
    "https://www.abs.gov.au/statistics/economy/government/"
    "taxation-revenue-australia/latest-release"
)

In [2]:
CHART_DIR = "./CHARTS/5506.0 - Taxation Revenue/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()

## Get data from ABS

In [3]:
tables = ra.grab_abs_url(url=TAX_URL)
do_prefixes = sorted({k.split('---')[0] for k in tables})
do_prefixes

['55060DO001_202425', '55060DO002_202425']

## Parse the cross-cutting summary table (DO002 Table 1)

Total taxation revenue broken down by level of government (Commonwealth, State, Local, All Levels) and tax category.

In [4]:
def locate_summary_prefix() -> str:
    # DO002 is the cross-cutting summary ("Total taxation revenue by level
    # of government and category" etc). Find it via its Contents page.
    for prefix in do_prefixes:
        contents = tables.get(f"{prefix}---Contents")
        if contents is None:
            continue
        for cell in contents.values.flatten():
            if not isinstance(cell, str):
                continue
            if "level of government and category" in cell.lower():
                return prefix
    raise KeyError("Could not locate summary DO file (DO002)")


SUMMARY_PREFIX = locate_summary_prefix()
SUMMARY_PREFIX

'55060DO002_202425'

In [5]:
def normalize_units(u: str) -> str:
    """Map ABS short unit strings (e.g. '$m') to the long form recalibrate() understands."""
    mapping = {
        "$m": "Dollars (Millions)",
        "$M": "Dollars (Millions)",
        "$b": "Dollars (Billions)",
        "$B": "Dollars (Billions)",
        "$": "Dollars",
    }
    return mapping.get(str(u).strip(), str(u))


# Level headers we recognise inside Table 1
LEVEL_HEADERS = {
    "Commonwealth government": "Commonwealth",
    "Total state government": "State",
    "Total local government": "Local",
    "All levels of government": "All Levels of Govt",
}

TAX_CATEGORIES = [
    "Taxes on income",
    "Employers payroll taxes",
    "Taxes on property",
    "Taxes on provision of goods and services",
    "Taxes on use of goods and performance of activities",
    "Total taxation revenue",
]


def parse_summary() -> tuple[dict[str, pd.DataFrame], list[pd.Period], str]:
    """Return (level -> DataFrame indexed by financial year, columns = tax categories; years; units)."""
    raw = tables[f"{SUMMARY_PREFIX}---Table_1"].dropna(how='all').dropna(axis=1, how='all')
    # header row = the one where col 1.. all look like '2015-16'
    header_row = None
    for idx, row in raw.iterrows():
        vals = [str(v) for v in row.values[1:] if isinstance(v, str)]
        if vals and all(len(v) == 7 and '-' in v for v in vals[:3]):
            header_row = idx
            break
    years_raw = raw.loc[header_row].iloc[1:].tolist()
    years = [pd.Period(str(int(y.split('-')[0]) + 1), freq='Y-JUN') for y in years_raw]
    units = normalize_units(str(raw.loc[header_row + 1].iloc[1]))
    # Walk rows, tracking which level block we're inside.
    per_level: dict[str, dict[str, list[float]]] = {v: {} for v in LEVEL_HEADERS.values()}
    current = None
    for _, row in raw.iloc[header_row + 2:].iterrows():
        label = row.iloc[0]
        if not isinstance(label, str):
            continue
        stripped = label.strip()
        matched_level = None
        for header, short in LEVEL_HEADERS.items():
            if stripped.lower() == header.lower():
                matched_level = short
                break
        if matched_level is not None:
            current = matched_level
            continue
        if current is None:
            continue
        if stripped in TAX_CATEGORIES:
            values = pd.to_numeric(row.iloc[1:], errors='coerce').tolist()
            per_level[current][stripped] = values
    frames = {level: pd.DataFrame(d, index=years).astype(float) for level, d in per_level.items() if d}
    return frames, years, units


frames, years, units = parse_summary()
print("units:", units, " years:", years[0], "..", years[-1])
{k: v.shape for k, v in frames.items()}

units: Dollars (Millions)  years: 2016 .. 2025


{'Commonwealth': (10, 6),
 'State': (10, 6),
 'Local': (10, 6),
 'All Levels of Govt': (10, 6)}

## Total taxation revenue: Commonwealth vs All Levels of Government

In [6]:
def plot_totals() -> None:
    df = pd.DataFrame({
        "Commonwealth": frames["Commonwealth"]["Total taxation revenue"],
        "All Levels of Govt": frames["All Levels of Govt"]["Total taxation revenue"],
    })
    df, u = ra.recalibrate(df, units)
    mg.line_plot_finalise(
        df,
        title="Total Taxation Revenue: Commonwealth vs All Levels",
        ylabel=u,
        xlabel="Financial Year ending 30 June",
        width=[2, 2],
        annotate=True,
        rounding=1,
        rfooter=RFOOTER,
        lfooter='Australia. "All Levels" = CW + State + Local, consolidated. ',
        legend={"loc": "best"},
        show=SHOW,
    )


plot_totals()

## Year-on-year growth in total taxation revenue

In [7]:
def plot_growth() -> None:
    df = pd.DataFrame({
        "Commonwealth": frames["Commonwealth"]["Total taxation revenue"].pct_change() * 100,
        "All Levels of Govt": frames["All Levels of Govt"]["Total taxation revenue"].pct_change() * 100,
    }).dropna(how='all')
    mg.line_plot_finalise(
        df,
        title="Total Taxation Revenue: Annual Growth",
        ylabel="Per cent change year on year",
        xlabel="Financial Year ending 30 June",
        width=[2, 2],
        y0=True,
        annotate=True,
        rounding=1,
        rfooter=RFOOTER,
        lfooter='Australia. "All Levels" = CW + State + Local, consolidated. ',
        legend={"loc": "best"},
        show=SHOW,
    )


plot_growth()

## Commonwealth taxation revenue by category

In [8]:
def plot_cw_by_category() -> None:
    cats = [c for c in TAX_CATEGORIES if c != "Total taxation revenue"]
    df = frames["Commonwealth"][cats].copy()
    # drop all-zero columns (e.g. Commonwealth has no property taxes)
    df = df.loc[:, (df != 0).any(axis=0)]
    df, u = ra.recalibrate(df, units)
    mg.line_plot_finalise(
        df,
        title="Commonwealth Taxation Revenue by Category",
        ylabel=u,
        xlabel="Financial Year ending 30 June",
        width=1.5,
        annotate=True,
        rounding=1,
        rfooter=RFOOTER,
        lfooter="Australia. Commonwealth Government. ",
        legend={"loc": "best", "fontsize": "x-small"},
        show=SHOW,
    )


plot_cw_by_category()

## All levels of government: taxation revenue by category

In [9]:
def plot_all_by_category() -> None:
    cats = [c for c in TAX_CATEGORIES if c != "Total taxation revenue"]
    df = frames["All Levels of Govt"][cats].copy()
    df = df.loc[:, (df != 0).any(axis=0)]
    df, u = ra.recalibrate(df, units)
    mg.line_plot_finalise(
        df,
        title="All Levels of Government: Taxation Revenue by Category",
        ylabel=u,
        xlabel="Financial Year ending 30 June",
        width=1.5,
        annotate=True,
        rounding=1,
        rfooter=RFOOTER,
        lfooter="Australia. All Levels of Government. ",
        legend={"loc": "best", "fontsize": "x-small"},
        show=SHOW,
    )


plot_all_by_category()

## Finished

In [10]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda
print("Done")

Last updated: 2026-04-21 13:57:48

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.12.0

conda environment: n/a

Compiler    : Clang 20.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.2.21
pandas : 3.0.2
readabs: 0.1.8

Watermark: 2.6.0

Done
